[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/E.%20Strategic%20Design%20%26%20Long-Term%20Investment%20%28Shaping%20the%20Future%29/42.%20The%20Peak%20Energy%20Consumption%20%26%20Load%20Shifting%20Problem/P42-Tier-2_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---

# 42. The Peak Energy Consumption and Load-Shifting Problem

## Tier 2: Greedy Randomized Adaptive Search Procedure (GRASP)

### Goal and scope

This notebook implements a GRASP heuristic for scheduling 15 shiftable operations over a 24-hour horizon. Each operation is assigned one start hour; its energy use is spread uniformly over its duration. The objective combines hourly energy charges and a charge based on the maximum hourly demand. The method constructs randomized greedy schedules, improves them with local search, and retains the best schedule found over multiple iterations.

The notebook reports a reference schedule in which every operation starts at hour 1, then compares its cost with the GRASP schedule. This reference is a cost benchmark, not a feasible GRASP schedule: it does not satisfy the start-time crowding rule. No mathematical-programming model or optimality comparison is implemented here, so the GRASP result is a heuristic solution, not a proven optimum.

### Model assumptions

- The horizon consists of 24 one-hour periods, numbered 1 through 24. An operation starting at hour \(s\) with duration \(d\) occupies hours \(s, s+1, \ldots, s+d-1\), and must finish within the horizon.
- Each operation's listed energy is its total energy in kWh. Its constant power while running is therefore \(E_i/d_i\) kW, where \(E_i\) is total energy and \(d_i\) is duration.
- Base demand is specified in MW. Since each period is one hour, hourly energy charges use the period's average power converted to kW and the price in dollars per kWh.
- A start-time crowding rule permits at most three operations to have start hours within one hour of one another. This rule limits clustered starts; it is not a general machine-capacity or non-overlap constraint.
- The storage parameters are included in the instance description but are not used by this scheduling heuristic. Storage charging, discharging, state of charge, and efficiency therefore do not affect the reported schedule or cost.
- The demand charge is applied once to the maximum hourly demand over this modeled horizon. No billing-period scaling, demand-response limits, precedence constraints, or operation-specific deadlines are modeled.

### GRASP in this notebook

GRASP repeatedly performs two phases. In the construction phase, it evaluates feasible operation/start-hour choices and forms a Restricted Candidate List (RCL) from the highest-scoring choices. A random choice from that list introduces controlled variation: a smaller \(\alpha\) favors the best candidates, while a larger \(\alpha\) admits more candidates. In the local-search phase, the algorithm tests pairwise exchanges of start times and individual start-time changes, retaining improving feasible schedules. The best result across the iterations is returned.

The construction score is a normalized estimate of how attractive a start hour is for an operation's energy charge, relative to that operation's feasible start hours. It guides construction; it is not the full schedule objective, which also includes the demand charge.

### Example instance

The instance has 15 operations over 24 hours. Operation durations range from 1 to 12 hours, and listed energy totals range from 80 to 480 kWh. The price is $0.15/kWh during hours 6–18 and $0.08/kWh during hours 1–5 and 19–24. The main GRASP run uses 100 iterations.

In [ ]:
import random
import time
from collections import Counter
from copy import deepcopy

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Fixed seeds make the randomized construction reproducible in a fresh run.
np.random.seed(42)
random.seed(42)

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

In [ ]:
class GRASPEnergyProblem:
    """Data for the 24-hour energy load-shifting instance."""

    def __init__(self):
        self.T = list(range(1, 25))

        self.operations = {
            "container_loading": {
                "duration": 4, "energy": 350, "window": list(range(1, 25))
            },
            "hvac_adjustment": {
                "duration": 2, "energy": 180, "window": list(range(1, 25))
            },
            "ev_fleet_charging": {
                "duration": 6, "energy": 480, "window": list(range(1, 25))
            },
            "warehouse_lighting": {
                "duration": 8, "energy": 120, "window": list(range(1, 25))
            },
            "crane_maintenance": {
                "duration": 3, "energy": 200, "window": list(range(1, 25))
            },
            "refrigeration_systems": {
                "duration": 5, "energy": 320, "window": list(range(1, 25))
            },
            "conveyor_systems": {
                "duration": 4, "energy": 280, "window": list(range(1, 25))
            },
            "security_lighting": {
                "duration": 12, "energy": 90, "window": list(range(1, 25))
            },
            "office_equipment": {
                "duration": 10, "energy": 150, "window": list(range(1, 25))
            },
            "pumping_systems": {
                "duration": 3, "energy": 240, "window": list(range(1, 25))
            },
            "ventilation_systems": {
                "duration": 4, "energy": 190, "window": list(range(1, 25))
            },
            "gate_operations": {
                "duration": 2, "energy": 110, "window": list(range(1, 25))
            },
            "yard_lighting": {
                "duration": 10, "energy": 140, "window": list(range(1, 25))
            },
            "admin_buildings": {
                "duration": 8, "energy": 160, "window": list(range(1, 25))
            },
            "emergency_systems": {
                "duration": 1, "energy": 80, "window": list(range(1, 25))
            },
        }

        self.prices = {
            t: (0.15 if 6 <= t <= 18 else 0.08)
            for t in self.T
        }

        # Base demand is in MW for each one-hour period.
        self.base_demand = {
            1: 8, 2: 7, 3: 6, 4: 7, 5: 9, 6: 12, 7: 15, 8: 18,
            9: 20, 10: 22, 11: 21, 12: 19, 13: 20, 14: 22,
            15: 21, 16: 19, 17: 17, 18: 15, 19: 13, 20: 11,
            21: 10, 22: 9, 23: 8, 24: 7,
        }

        # Demand charge in dollars per kW of maximum hourly demand.
        self.demand_charge = 18

        # Storage is recorded as instance data but is outside this heuristic model.
        self.storage = {
            "capacity": 500,      # kWh
            "rate": 100,          # kW charge/discharge rate
            "efficiency": 0.90,  # round-trip efficiency
        }


grasp_problem = GRASPEnergyProblem()
peak_hours = sum(
    grasp_problem.prices[t] == max(grasp_problem.prices.values())
    for t in grasp_problem.T
)
off_peak_hours = len(grasp_problem.T) - peak_hours

print(
    f"GRASP problem initialized with {len(grasp_problem.operations)} "
    f"operations and {len(grasp_problem.T)} time periods."
)
print(f"Peak-price hours: {peak_hours}")
print(f"Off-peak-price hours: {off_peak_hours}")

## GRASP construction, local search, and objective

A solution is a mapping \(x_i\) from each operation \(i\) to its start hour. The construction phase adds one operation at a time. At each step it enumerates feasible operation/start-hour candidates, scores them, and randomly selects from the RCL. The local-search neighborhood consists of pairwise exchanges of operation start hours and single-operation changes to another permitted start hour.

For a schedule \(x\), let \(P_t(x)\) be the total power during hour \(t\), in kW, including base demand and active operations. The schedule cost is

$$
C(x) = \sum_{t \in T} p_t P_t(x) + c_D \max_{t \in T} P_t(x),
$$

where \(T\) is the set of hours, \(p_t\) is the energy price in dollars per kWh, and \(c_D\) is the demand-charge coefficient in dollars per kW. Because each period lasts one hour, \(p_t P_t(x)\) is the energy charge for that period. The second term charges the maximum hourly demand once over the modeled horizon.

For operation \(i\), with total energy \(E_i\), duration \(d_i\), and start time \(x_i\), its power contribution is \(E_i/d_i\) kW in each occupied hour. Thus,

$$
P_t(x) = 1000 B_t + \sum_{i \in O} \frac{E_i}{d_i}
\\mathbb{1}\{x_i \le t \le x_i+d_i-1\},
$$

where \(B_t\) is base demand in MW, \(O\) is the set of operations, and \(\mathbb{1}\{\cdot\}\) equals 1 when its condition is true and 0 otherwise. In the code, `calculate_hourly_demand` builds \(P_t(x)\), and `calculate_solution_cost` evaluates \(C(x)\).

The RCL score uses the operation's energy charge at a candidate start hour. If \(g_i(s)\) is that charge, and \(g_i^{\min}\) and \(g_i^{\max}\) are the minimum and maximum charges over its feasible individual start hours, its normalized attractiveness is

$$
q_i(s) = \begin{cases}
\\dfrac{g_i^{\max}-g_i(s)}{g_i^{\max}-g_i^{\min}}, & g_i^{\max} > g_i^{\min}, \\
0, & g_i^{\max} = g_i^{\min}.
\end{cases}
$$

The RCL contains candidates satisfying \(q_i(s) \ge q_{\max} - \alpha(q_{\max}-q_{\min})\), where \(q_{\max}\) and \(q_{\min}\) are the largest and smallest scores in the current candidate set and \(\alpha \in [0,1]\). The code uses `alpha` for this threshold and selects uniformly at random from the resulting list. Scores guide construction only; local search evaluates the full objective, including the demand charge.

This implementation explicitly enumerates candidate start hours. If there are \(|O|\) operations and \(|T|\) possible start hours, one construction pass considers at most \(|O||T|\) operation/start-hour pairs per scheduling step. Local search tests pairwise exchanges and individual start-time changes and may repeat these neighborhoods until no tested move improves the current schedule; its work therefore depends on the instance and the number of improvement passes.

In [ ]:
class GRASPOptimizer:
    """GRASP heuristic for the energy load-shifting instance."""

    def __init__(self, problem, alpha=0.25, max_iterations=100):
        if not 0 <= alpha <= 1:
            raise ValueError("alpha must be between 0 and 1.")
        if max_iterations < 1:
            raise ValueError("max_iterations must be at least 1.")

        self.problem = problem
        self.alpha = alpha
        self.max_iterations = max_iterations
        self.best_solution = None
        self.best_cost = float("inf")
        self.iteration_costs = []

    def calculate_savings_potential(self, op_name, start_time):
        """Return normalized energy-cost attractiveness for one start hour."""
        op_data = self.problem.operations[op_name]
        duration = op_data["duration"]
        energy_rate = op_data["energy"] / duration
        feasible_starts = [
            start
            for start in op_data["window"]
            if start + duration - 1 <= self.problem.T[-1]
        ]

        def energy_charge(start):
            occupied_hours = range(start, start + duration)
            return sum(
                self.problem.prices[hour] * energy_rate
                for hour in occupied_hours
            )

        charges = [energy_charge(start) for start in feasible_starts]
        current_charge = energy_charge(start_time)
        best_charge = min(charges)
        worst_charge = max(charges)

        if worst_charge > best_charge:
            return (worst_charge - current_charge) / (worst_charge - best_charge)
        return 0.0

    def construct_greedy_solution(self):
        """Build a feasible schedule using a randomized greedy RCL."""
        solution = {}
        remaining_operations = list(self.problem.operations)

        while remaining_operations:
            candidates = []

            for op_name in remaining_operations:
                for start_time in self.problem.operations[op_name]["window"]:
                    if self.is_feasible_schedule(solution, op_name, start_time):
                        score = self.calculate_savings_potential(op_name, start_time)
                        candidates.append((op_name, start_time, score))

            if not candidates:
                raise RuntimeError(
                    "No feasible operation/start-hour candidate remains."
                )

            scores = [candidate[2] for candidate in candidates]
            max_score = max(scores)
            min_score = min(scores)
            threshold = max_score - self.alpha * (max_score - min_score)
            rcl = [candidate for candidate in candidates if candidate[2] >= threshold]

            op_name, start_time, _ = random.choice(rcl)
            solution[op_name] = start_time
            remaining_operations.remove(op_name)

        return solution

    def is_feasible_schedule(self, current_solution, op_name, start_time):
        """Check the horizon, allowed window, and start-time crowding rule."""
        op_data = self.problem.operations[op_name]
        duration = op_data["duration"]

        if start_time not in op_data["window"]:
            return False
        if start_time + duration - 1 > self.problem.T[-1]:
            return False

        nearby_starts = sum(
            abs(scheduled_start - start_time) <= 1
            for scheduled_start in current_solution.values()
        )
        # The candidate itself makes the maximum permitted cluster size three.
        return nearby_starts <= 2

    def local_search(self, solution):
        """Improve a schedule with start-time exchanges and individual moves."""
        best_local_solution = deepcopy(solution)
        best_local_cost = self.calculate_solution_cost(best_local_solution)
        improved = True

        while improved:
            improved = False
            operations = list(best_local_solution)

            # Exchanging two start times explores pairwise schedule changes.
            for i in range(len(operations)):
                for j in range(i + 1, len(operations)):
                    op1, op2 = operations[i], operations[j]
                    new_solution = deepcopy(best_local_solution)
                    new_solution[op1], new_solution[op2] = (
                        new_solution[op2], new_solution[op1]
                    )

                    if self.is_solution_feasible(new_solution):
                        new_cost = self.calculate_solution_cost(new_solution)
                        if new_cost < best_local_cost - 1e-9:
                            best_local_solution = new_solution
                            best_local_cost = new_cost
                            improved = True

            # Individual moves allow a start time to change without an exchange.
            for op_name in operations:
                current_start = best_local_solution[op_name]
                for new_start in self.problem.operations[op_name]["window"]:
                    if new_start == current_start:
                        continue

                    new_solution = deepcopy(best_local_solution)
                    new_solution[op_name] = new_start

                    if self.is_solution_feasible(new_solution):
                        new_cost = self.calculate_solution_cost(new_solution)
                        if new_cost < best_local_cost - 1e-9:
                            best_local_solution = new_solution
                            best_local_cost = new_cost
                            improved = True
                            break

        return best_local_solution

    def is_solution_feasible(self, solution):
        """Check that all operations are scheduled and all rules are met."""
        if set(solution) != set(self.problem.operations):
            return False

        for op_name, start_time in solution.items():
            other_starts = {
                other_name: other_start
                for other_name, other_start in solution.items()
                if other_name != op_name
            }
            if not self.is_feasible_schedule(other_starts, op_name, start_time):
                return False
        return True

    def calculate_hourly_demand(self, solution):
        """Return total hourly demand in kW for the supplied schedule."""
        hourly_demand = {}

        for hour in self.problem.T:
            demand_kw = self.problem.base_demand[hour] * 1000

            for op_name, start_time in solution.items():
                op_data = self.problem.operations[op_name]
                end_time = start_time + op_data["duration"] - 1
                if start_time <= hour <= end_time:
                    demand_kw += op_data["energy"] / op_data["duration"]

            hourly_demand[hour] = demand_kw

        return hourly_demand

    def calculate_solution_cost(self, solution):
        """Evaluate energy charges plus the maximum-demand charge."""
        hourly_demand = self.calculate_hourly_demand(solution)
        energy_cost = sum(
            self.problem.prices[hour] * hourly_demand[hour]
            for hour in self.problem.T
        )
        peak_demand_kw = max(hourly_demand.values())
        demand_cost = self.problem.demand_charge * peak_demand_kw
        return energy_cost + demand_cost

    def run_grasp(self):
        """Run construction and local search for the configured iterations."""
        self.best_solution = None
        self.best_cost = float("inf")
        self.iteration_costs = []

        print(f"Running GRASP with {self.max_iterations} iterations...")

        for iteration in range(self.max_iterations):
            solution = self.construct_greedy_solution()
            improved_solution = self.local_search(solution)
            cost = self.calculate_solution_cost(improved_solution)

            if cost < self.best_cost:
                self.best_cost = cost
                self.best_solution = improved_solution

            self.iteration_costs.append(cost)

            if (iteration + 1) % 10 == 0:
                print(
                    f"Iteration {iteration + 1}: best cost so far = "
                    f"${self.best_cost:,.2f}; current cost = ${cost:,.2f}"
                )

        print("\nGRASP completed.")
        print(f"Best schedule cost: ${self.best_cost:,.2f}")
        return self.best_solution, self.best_cost


grasp_optimizer = GRASPOptimizer(
    grasp_problem,
    alpha=0.25,
    max_iterations=100,
)
print("GRASP optimizer initialized.")

## Run GRASP and inspect the schedule

The next cell performs the 100 construction-and-improvement iterations. It reports the best schedule found, not a guaranteed global optimum. The displayed price label describes the price at the operation's start hour; an operation may continue into hours with a different price.

In [ ]:
start_clock = time.time()
best_solution, best_cost = grasp_optimizer.run_grasp()
elapsed_seconds = time.time() - start_clock

assert grasp_optimizer.is_solution_feasible(best_solution)
assert np.isclose(
    grasp_optimizer.calculate_solution_cost(best_solution),
    best_cost,
)

print(f"\nExecution time: {elapsed_seconds:.2f} seconds")
print(f"Best schedule cost found: ${best_cost:,.2f}")
print("\n=== BEST GRASP SCHEDULE FOUND ===")

for op_name, start_hour in sorted(best_solution.items(), key=lambda item: item[1]):
    op_data = grasp_problem.operations[op_name]
    end_hour = start_hour + op_data["duration"] - 1
    price_type = (
        "Peak price"
        if grasp_problem.prices[start_hour] == max(grasp_problem.prices.values())
        else "Off-peak price"
    )
    print(
        f"{op_name.replace('_', ' ').title():<25}: "
        f"Hours {start_hour:2d}-{end_hour:2d} "
        f"({op_data['duration']} h, {op_data['energy']} kWh total) "
        f"[{price_type at start}]"
    )

## Compare against an unshifted reference schedule

The reference assigns every operation to start at hour 1. It provides a transparent cost baseline, but it violates the start-time crowding rule and is not a feasible GRASP solution. Cost differences should therefore be interpreted as comparisons with this reference schedule, not as a feasible-versus-feasible performance guarantee.

In [ ]:
def calculate_baseline_comparison(problem, optimizer):
    """Calculate the cost of the reference schedule with all starts at hour 1."""
    baseline_solution = {op_name: 1 for op_name in problem.operations}
    baseline_cost = optimizer.calculate_solution_cost(baseline_solution)
    return baseline_solution, baseline_cost


def calculate_peak_demand(problem, solution):
    """Return maximum hourly demand in MW for a schedule."""
    optimizer = GRASPOptimizer(problem, max_iterations=1)
    hourly_demand_kw = optimizer.calculate_hourly_demand(solution)
    return max(hourly_demand_kw.values()) / 1000


baseline_solution, baseline_cost = calculate_baseline_comparison(
    grasp_problem,
    grasp_optimizer,
)
baseline_peak = calculate_peak_demand(grasp_problem, baseline_solution)
grasp_peak = calculate_peak_demand(grasp_problem, best_solution)
cost_savings = baseline_cost - best_cost
peak_reduction = baseline_peak - grasp_peak

print("\n=== REFERENCE COST COMPARISON ===")
print(f"Unshifted reference cost: ${baseline_cost:,.2f}")
print(f"Best GRASP schedule cost: ${best_cost:,.2f}")
print(f"Cost difference:          ${cost_savings:,.2f}")
print(f"Difference as percentage of reference: {cost_savings / baseline_cost * 100:.2f}%")

print("\n=== PEAK-DEMAND COMPARISON ===")
print(f"Unshifted reference peak demand: {baseline_peak:.3f} MW")
print(f"Best GRASP schedule peak demand: {grasp_peak:.3f} MW")
print(f"Peak-demand difference:          {peak_reduction:.3f} MW")
print(f"Difference as percentage of reference: {peak_reduction / baseline_peak * 100:.2f}%")

## Visualize the search and schedule

The convergence panel shows each iteration's locally improved schedule cost and the best cost observed up to that iteration. The demand panel compares hourly profiles with the unshifted reference. The remaining panels summarize operation start hours and total schedule costs. The cost chart inherits the feasibility limitation of the reference schedule described above.

In [ ]:
def create_grasp_visualizations(problem, optimizer, baseline_solution, baseline_cost):
    """Plot search progress, hourly demand, start times, and schedule costs."""
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle("GRASP Energy Load-Shifting Results", fontsize=16, fontweight="bold")

    # The running minimum distinguishes best-so-far progress from iteration variation.
    ax1 = axes[0, 0]
    iterations = np.arange(1, len(optimizer.iteration_costs) + 1)
    ax1.plot(
        iterations,
        optimizer.iteration_costs,
        color="tab:blue",
        linewidth=1,
        alpha=0.7,
        label="Locally improved schedule",
    )
    ax1.plot(
        iterations,
        np.minimum.accumulate(optimizer.iteration_costs),
        color="tab:red",
        linewidth=2,
        label="Best so far",
    )
    ax1.set_xlabel("GRASP iteration")
    ax1.set_ylabel("Schedule cost ($)")
    ax1.set_title("Search Progress")
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # Demand profiles are computed from the same power-balance calculation as the objective.
    ax2 = axes[0, 1]
    hours = problem.T
    demand_calculator = GRASPOptimizer(problem, max_iterations=1)
    baseline_profile = [
        demand_calculator.calculate_hourly_demand(baseline_solution)[hour] / 1000
        for hour in hours
    ]
    grasp_profile = [
        demand_calculator.calculate_hourly_demand(optimizer.best_solution)[hour] / 1000
        for hour in hours
    ]
    ax2.plot(hours, baseline_profile, "o-", linewidth=1.8, markersize=4, label="Unshifted reference")
    ax2.plot(hours, grasp_profile, "s-", linewidth=1.8, markersize=4, label="Best GRASP schedule")
    ax2.fill_between(
        hours,
        baseline_profile,
        grasp_profile,
        color="tab:green",
        alpha=0.2,
        label="Profile difference",
    )
    ax2.set_xlabel("Hour")
    ax2.set_ylabel("Power demand (MW)")
    ax2.set_title("Hourly Demand Profile")
    ax2.set_xticks(hours[::2])
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    ax3 = axes[1, 0]
    start_times = list(optimizer.best_solution.values())
    ax3.hist(
        start_times,
        bins=np.arange(0.5, 25.5, 1),
        color="tab:blue",
        alpha=0.75,
        edgecolor="black",
    )
    ax3.axvspan(5.5, 18.5, color="tab:red", alpha=0.15, label="Peak-price hours")
    ax3.set_xlabel("Operation start hour")
    ax3.set_ylabel("Number of operations")
    ax3.set_title("Distribution of Start Hours")
    ax3.set_xticks(range(1, 25, 2))
    ax3.legend()
    ax3.grid(True, alpha=0.3)

    ax4 = axes[1, 1]
    grasp_cost = optimizer.best_cost
    labels = ["Unshifted reference", "Best GRASP schedule"]
    costs = [baseline_cost, grasp_cost]
    bars = ax4.bar(labels, costs, color=["tab:red", "tab:blue"], alpha=0.75)
    ax4.set_ylabel("Schedule cost ($)")
    ax4.set_title("Cost Comparison")
    ax4.tick_params(axis="x", rotation=10)
    for bar, cost in zip(bars, costs):
        ax4.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f"${cost:,.0f}",
            ha="center",
            va="bottom",
            fontweight="bold",
        )
    ax4.grid(True, axis="y", alpha=0.3)

    fig.tight_layout()
    plt.show()

    total_energy = sum(op["energy"] for op in problem.operations.values())
    average_duration = np.mean(
        [op["duration"] for op in problem.operations.values()]
    )
    reference_peak = calculate_peak_demand(problem, baseline_solution)
    schedule_peak = calculate_peak_demand(problem, optimizer.best_solution)
    savings = baseline_cost - optimizer.best_cost

    analysis_data = {
        "Metric": [
            "Number of operations",
            "Total operation energy (kWh)",
            "Average operation duration (h)",
            "Peak-price hours",
            "Off-peak-price hours",
            "Unshifted reference cost ($)",
            "Best GRASP schedule cost ($)",
            "Cost difference ($)",
            "Cost difference (%)",
            "Reference peak demand (MW)",
            "GRASP peak demand (MW)",
            "Peak-demand difference (MW)",
        ],
        "Value": [
            len(problem.operations),
            total_energy,
            f"{average_duration:.2f}",
            sum(problem.prices[t] == max(problem.prices.values()) for t in problem.T),
            sum(problem.prices[t] == min(problem.prices.values()) for t in problem.T),
            f"{baseline_cost:,.2f}",
            f"{optimizer.best_cost:,.2f}",
            f"{savings:,.2f}",
            f"{savings / baseline_cost * 100:.2f}%",
            f"{reference_peak:.3f}",
            f"{schedule_peak:.3f}",
            f"{reference_peak - schedule_peak:.3f}",
        ],
    }

    print("\n=== INSTANCE AND SCHEDULE SUMMARY ===")
    print(pd.DataFrame(analysis_data).to_string(index=False))


create_grasp_visualizations(
    grasp_problem,
    grasp_optimizer,
    baseline_solution,
    baseline_cost,
)

## Solution quality and run-to-run consistency

GRASP uses random choices during construction, so independent executions can return different schedules and costs. The following experiment runs ten additional optimizers with 50 iterations each and summarizes their costs together with the main run. The experiment describes behavior on this instance; it does not establish a general performance guarantee. The fixed random seeds make a fresh execution reproducible when the same Python and library behavior is used.

In [ ]:
def analyze_solution_quality(problem, optimizer, num_runs=10):
    """Summarize costs and operation start-time consistency across runs."""
    if num_runs < 1:
        raise ValueError("num_runs must be at least 1.")

    print("\n=== SOLUTION QUALITY ANALYSIS ===")
    print(f"Running {num_runs} additional GRASP executions.")

    additional_costs = []
    additional_solutions = []

    for run in range(num_runs):
        test_optimizer = GRASPOptimizer(
            problem,
            alpha=optimizer.alpha,
            max_iterations=50,
        )
        solution, cost = test_optimizer.run_grasp()
        additional_costs.append(cost)
        additional_solutions.append(solution)

        if run < 3:
            print(f"Additional run {run + 1}: best cost = ${cost:,.2f}")

    all_costs = [optimizer.best_cost] + additional_costs
    mean_cost = float(np.mean(all_costs))
    std_cost = float(np.std(all_costs))
    min_cost = float(np.min(all_costs))
    max_cost = float(np.max(all_costs))

    print("\n=== COST SUMMARY ACROSS RUNS ===")
    print(f"Best observed cost: ${min_cost:,.2f}")
    print(f"Mean cost:          ${mean_cost:,.2f}")
    print(f"Standard deviation: ${std_cost:,.2f}")
    print(f"Highest cost:       ${max_cost:,.2f}")
    if not np.isclose(mean_cost, 0.0):
        print(f"Coefficient of variation: {std_cost / mean_cost * 100:.2f}%")

    operation_consistency = {}
    all_solutions = additional_solutions + [optimizer.best_solution]

    for op_name in problem.operations:
        start_times = [solution[op_name] for solution in all_solutions]
        most_common_start, frequency = Counter(start_times).most_common(1)[0]
        operation_consistency[op_name] = {
            "most_common_start": most_common_start,
            "consistency_pct": frequency / len(start_times) * 100,
            "start_times": start_times,
        }

    sorted_consistency = sorted(
        operation_consistency.items(),
        key=lambda item: (-item[1]["consistency_pct"], item[0]),
    )

    print("\n=== MOST CONSISTENT OPERATION START HOURS ===")
    for rank, (op_name, data) in enumerate(sorted_consistency[:5], start=1):
        print(
            f"{rank}. {op_name.replace('_', ' ').title():<25}: "
            f"most frequent start hour {data['most_common_start']} "
            f"({data['consistency_pct']:.1f}% of runs)"
        )

    return operation_consistency


consistency_data = analyze_solution_quality(
    grasp_problem,
    grasp_optimizer,
    num_runs=10,
)

## Interpretation and limitations

The best schedule found is the lowest-cost feasible schedule encountered by the specified GRASP runs. It is not an optimality certificate: the construction is randomized, and the local search examines only its defined exchange and relocation neighborhood. The convergence plot records search progress for one run, while the repeated-run summary describes variability for the chosen seed sequence and iteration counts.

The reference comparison is useful for illustrating the effect of scheduling, but its all-at-hour-1 schedule violates the start-time crowding rule. It should not be interpreted as a comparison between two feasible schedules. Results also depend on the model's objective and assumptions: the simplified crowding rule is not a physical equipment-capacity model, storage is inactive, and the demand charge is applied once to the maximum demand in the 24-hour horizon.

### Strengths and limitations of this tier

**Strengths**
- GRASP uses randomized greedy construction followed by local improvement, making the search logic explicit and adaptable within the same heuristic framework.
- The implementation requires no specialized optimization solver.
- Multiple iterations explore different constructed schedules, and the best feasible schedule found is retained.

**Limitations**
- A heuristic does not guarantee a globally optimal schedule or provide an optimality gap.
- Results can depend on the random sequence, the RCL parameter \(\alpha\), and the iteration limit.
- The local-search neighborhood may not connect all feasible schedules through improving moves.
- Candidate enumeration and repeated local-search evaluations can become costly as the number of operations and allowed start hours grow; this implementation does not establish linear scalability.
- The model omits storage operation, detailed equipment capacities, precedence relations, and other operational constraints that may matter in practice.

Within this scope, the notebook demonstrates the GRASP solution method for energy load shifting and provides diagnostics for its schedule cost, demand profile, and run-to-run behavior.